# RDD2022 YOLOv11
Dieses Notebook trainiert ein YOLOv11n-Modell (nano, optimiert) auf dem RDD2022-Datensatz und visualisiert alle relevanten Metriken. Training dauert ca. 2-3 Stunden auf RTX 3050.

## 1. Setup & Imports

In [1]:
print("test")

test


In [ ]:
import os
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
from ultralytics import YOLO

BASE       = Path('.').resolve()
DATA_NINJA = BASE / 'data' / 'dataset-ninja'
DATA_YOLO  = BASE / 'data' / 'yolo'
YAML       = DATA_YOLO / 'rdd2022.yaml'
RUNS_DIR   = BASE / 'runs'
RUN_NAME   = 'rdd2022_yolo11n'  # ← Angepasst auf nano-Modell

CLASSES = [
    'alligator crack', 'block crack', 'longitudinal crack',
    'other corruption', 'pothole', 'repair', 'transverse crack'
]
COLORS = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c']

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'YAML ok  : {YAML.exists()}')

PyTorch  : 2.4.1+cu124
CUDA     : True
GPU      : NVIDIA GeForce RTX 3050
VRAM     : 8.6 GB
YAML ok  : True


## 2. Datensatz-Analyse

In [3]:
splits = {}
for split in ['train', 'val', 'test']:
    img_dir = DATA_YOLO / 'images' / split
    splits[split] = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0

for s, n in splits.items():
    print(f'  {s:6s}: {n:6,} Bilder')
print(f'  TOTAL : {sum(splits.values()):6,} Bilder')

  train : 30,708 Bilder
  val   :  7,677 Bilder
  test  :  9,035 Bilder
  TOTAL : 47,420 Bilder


In [5]:
counter = Counter()
empty_count = 0
for ann_path in (DATA_NINJA / 'train' / 'ann').iterdir():
    data = json.loads(ann_path.read_text())
    objs = data.get('objects', [])
    if not objs:
        empty_count += 1
    for obj in objs:
        counter[obj['classTitle']] += 1

total_boxes = sum(counter.values())
print(f'Bounding Boxes gesamt: {total_boxes:,}  |  Leere Bilder: {empty_count:,}')
print()
for cls in CLASSES:
    n = counter.get(cls, 0)
    pct = n / max(total_boxes, 1) * 100
    bar = chr(9608) * int(pct / 2)
    print(f'  {cls:22s}  {n:6,}  ({pct:4.1f}%)  {bar}')

Bounding Boxes gesamt: 61,082  |  Leere Bilder: 12,658

  alligator crack         10,617  (17.4%)  ████████
  block crack                  0  ( 0.0%)  
  longitudinal crack      26,196  (42.9%)  █████████████████████
  other corruption         5,850  ( 9.6%)  ████
  pothole                  6,544  (10.7%)  █████
  repair                       0  ( 0.0%)  
  transverse crack        11,875  (19.4%)  █████████


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.bar(list(splits.keys()), list(splits.values()), color=['#3498db','#e67e22','#2ecc71'])
for i, (k, v) in enumerate(splits.items()):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=10)
ax.set_title('Datensatz-Splits', fontsize=13, fontweight='bold')
ax.set_ylabel('Anzahl Bilder')
ax.set_ylim(0, max(splits.values()) * 1.15)

ax = axes[1]
vals = [counter.get(c, 0) for c in CLASSES]
bars = ax.barh(CLASSES, vals, color=COLORS)
for bar, v in zip(bars, vals):
    ax.text(v + 50, bar.get_y() + bar.get_height() / 2, f'{v:,}', va='center', fontsize=9)
ax.set_title('Klassenverteilung (Train)', fontsize=13, fontweight='bold')
ax.set_xlabel('Anzahl Bounding Boxes')

ax = axes[2]
ax.pie(vals, labels=CLASSES, colors=COLORS, autopct='%1.1f%%', startangle=140,
       textprops={'fontsize': 8})
ax.set_title('Klassenanteil', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('dataset_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert: dataset_stats.png')

Gespeichert: dataset_stats.png


C:\Users\phili\AppData\Local\Temp\ipykernel_15952\2503716414.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
img_dir = DATA_NINJA / 'train' / 'img'
ann_dir = DATA_NINJA / 'train' / 'ann'
cls_to_color = dict(zip(CLASSES, COLORS))

samples = []
for ann_path in sorted(ann_dir.iterdir()):
    data = json.loads(ann_path.read_text())
    if data.get('objects'):
        samples.append(ann_path)
    if len(samples) >= 6:
        break

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, ann_path in zip(axes.flat, samples):
    img = Image.open(img_dir / ann_path.stem)
    ax.imshow(img)
    data = json.loads(ann_path.read_text())
    for obj in data['objects']:
        cls = obj['classTitle']
        x1, y1 = obj['points']['exterior'][0]
        x2, y2 = obj['points']['exterior'][1]
        color = cls_to_color.get(cls, 'white')
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1, max(y1-4, 0), cls, color=color, fontsize=7,
                bbox=dict(facecolor='black', alpha=0.5, pad=1))
    tag = data['tags'][0]['name'] if data.get('tags') else ''
    ax.set_title(f'{ann_path.stem}\n{tag}', fontsize=8)
    ax.axis('off')

plt.suptitle('Beispielbilder mit Ground-Truth Annotationen', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert: sample_images.png')

Gespeichert: sample_images.png


C:\Users\phili\AppData\Local\Temp\ipykernel_15952\480862250.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Training

In [10]:
MODEL_PATH = 'yolo11n.pt'  # ← Kleiner: Nano statt Medium (schneller, weniger VRAM)

TRAIN_ARGS = dict(
    data        = str(YAML.as_posix()),
    epochs      = 20,             # ← Reduziert von 50 → ~2-3 Std. Trainingsdauer
    imgsz       = 384,            # ← Optimiert für RTX 3050 (schneller als 640)
    batch       = 16,             # ← Erhöht von 8 (Sanity Check war erfolgreich)
    workers     = 0,
    device      = 0,
    project     = str(RUNS_DIR.as_posix()),
    name        = RUN_NAME,
    patience    = 5,              # ← Early stopping nach 5 Epochen ohne Verbesserung
    save_period = 5,              # ← Weniger häufiges Speichern
    val         = True,
    plots       = True,
    exist_ok    = True,
    optimizer   = 'AdamW',
    lr0         = 0.001,
    verbose     = True,
)

print('Training-Konfiguration (optimiert für RTX 3050, ~2-3 Std):')
for k, v in TRAIN_ARGS.items():
    print(f'  {k:15s}: {v}')

Training-Konfiguration (optimiert für RTX 3050, ~2-3 Std):
  data           : C:/Users/phili/Desktop/OTH/Master/fpp/hsp/hsp-1/Python/data/yolo/rdd2022.yaml
  epochs         : 20
  imgsz          : 384
  batch          : 16
  workers        : 0
  device         : 0
  project        : C:/Users/phili/Desktop/OTH/Master/fpp/hsp/hsp-1/Python/runs
  name           : rdd2022_yolo11m
  patience       : 5
  save_period    : 5
  val            : True
  plots          : True
  exist_ok       : True
  optimizer      : AdamW
  lr0            : 0.001
  verbose        : True


In [9]:
# SANITY CHECK: Schneller Test vor dem langen Training (ca. 3-5 min)
print("=" * 60)
print("SANITY CHECK: Kurzer Test mit Memory-optimierter Konfiguration")
print("=" * 60)

SANITY_CHECK = True  # ← Setze auf False um zu überspringen

if SANITY_CHECK:
    print("\n⏱️  Starte Sanity Check (dauert ca. 3-5 min)...\n")
    
    model_sanity = YOLO(MODEL_PATH)
    sanity_result = model_sanity.train(
        data        = str(YAML.as_posix()),
        epochs      = 1,           # ← Nur 1 Epoche
        imgsz       = 384,         # ← Optimiert für RTX 3050 (8GB)
        batch       = 8,           # ← Reduziert (RTX 3050 Memory-Limit)
        workers     = 0,
        device      = 0,
        project     = str(RUNS_DIR.as_posix()),
        name        = 'sanity_check',
        val         = True,
        plots       = False,
        exist_ok    = True,
        verbose     = True,
    )
    print("\n✅ SANITY CHECK ERFOLGREICH!")
    print(f"   → Setup funktioniert")
    print(f"   → Starte nun full training mit: imgsz=640, batch=16")
else:
    print("\n⏭️  Sanity Check übersprungen (SANITY_CHECK=False)")


SANITY CHECK: Kurzer Test mit Memory-optimierter Konfiguration

⏱️  Starte Sanity Check (dauert ca. 3-5 min)...

New https://pypi.org/project/ultralytics/8.4.50 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.11.15 torch-2.4.1+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/phili/Desktop/OTH/Master/fpp/hsp/hsp-1/Python/data/yolo/rdd2022.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, 

In [11]:
model = YOLO(MODEL_PATH)
train_results = model.train(**TRAIN_ARGS)
print(f'\nTraining abgeschlossen.')
#print(f'Beste Gewichte: {train_results.save_dir}/weights/best.pt')

New https://pypi.org/project/ultralytics/8.4.50 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.11.15 torch-2.4.1+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/phili/Desktop/OTH/Master/fpp/hsp/hsp-1/Python/data/yolo/rdd2022.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n

## 4. Ergebnisse & Metriken

In [12]:
results_csv = RUNS_DIR / RUN_NAME / 'results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
print(f'Epochen trainiert : {len(df)}')
print(f'Spalten           : {list(df.columns)}')
print('\nLetzte Epoche:')
print(df.iloc[-1].to_string())

Epochen trainiert : 20
Spalten           : ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

Letzte Epoche:
epoch                      20.00000
time                    16992.20000
train/box_loss              1.77119
train/cls_loss              1.84147
train/dfl_loss              1.46914
metrics/precision(B)        0.52159
metrics/recall(B)           0.43170
metrics/mAP50(B)            0.43127
metrics/mAP50-95(B)         0.21518
val/box_loss                1.89300
val/cls_loss                1.98548
val/dfl_loss                1.45996
lr/pg0                      0.00006
lr/pg1                      0.00006
lr/pg2                      0.00006


In [13]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
ep = df.get('epoch', pd.Series(range(len(df))))

loss_cfg = [
    ('train/box_loss', 'Box Loss',   '#e74c3c'),
    ('train/cls_loss', 'Class Loss', '#e67e22'),
    ('train/dfl_loss', 'DFL Loss',   '#9b59b6'),
]
for ax, (col, title, color) in zip(axes[0], loss_cfg):
    if col in df.columns:
        ax.plot(ep, df[col], color=color, linewidth=2, label='train')
        vcol = col.replace('train/', 'val/')
        if vcol in df.columns:
            ax.plot(ep, df[vcol], color=color, linewidth=2, linestyle='--', alpha=0.7, label='val')
        ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoche')
        ax.legend(); ax.grid(alpha=0.3)

metric_cfg = [
    ('metrics/mAP50(B)',    'mAP@50',            '#3498db'),
    ('metrics/mAP50-95(B)', 'mAP@50-95',         '#2ecc71'),
    ('metrics/precision(B)','Precision & Recall', '#1abc9c'),
]
for ax, (col, title, color) in zip(axes[1], metric_cfg):
    if col in df.columns:
        lbl = col.split('/')[1].split('(')[0]
        ax.plot(ep, df[col], color=color, linewidth=2, label=lbl)
        if col == 'metrics/precision(B)' and 'metrics/recall(B)' in df.columns:
            ax.plot(ep, df['metrics/recall(B)'], color='#e74c3c', linewidth=2, label='recall')
        bi = df[col].idxmax()
        ax.axvline(ep.iloc[bi], color='gray', linestyle=':', alpha=0.7)
        ax.annotate(f'Best: {df[col].iloc[bi]:.3f}',
                    xy=(ep.iloc[bi], df[col].iloc[bi]),
                    xytext=(10, -15), textcoords='offset points', fontsize=9)
        ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoche')
        ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Trainings-Metriken RDD2022 YOLOv11m', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert: training_curves.png')

Gespeichert: training_curves.png


C:\Users\phili\AppData\Local\Temp\ipykernel_15952\3874169786.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
run_dir = RUNS_DIR / RUN_NAME
plot_files = [
    ('confusion_matrix_normalized.png', 'Confusion Matrix (norm.)'),
    ('PR_curve.png',  'Precision-Recall'),
    ('F1_curve.png',  'F1-Score'),
]
available = [(f, t) for f, t in plot_files if (run_dir / f).exists()]
if available:
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 6))
    if len(available) == 1:
        axes = [axes]
    for ax, (fname, title) in zip(axes, available):
        ax.imshow(Image.open(run_dir / fname))
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Keine Plots gefunden in:', run_dir)

C:\Users\phili\AppData\Local\Temp\ipykernel_15952\3981079718.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Evaluation auf Test-Set

In [15]:
best_weights = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
print(f'Lade: {best_weights}')
model_eval = YOLO(str(best_weights))

metrics = model_eval.val(
    data=str(YAML.as_posix()),
    split='test',
    device=0,
    plots=True,
    verbose=True,
)

print('\n=== Test-Set Ergebnisse ===')
print(f'mAP@50      : {metrics.box.map50:.4f}')
print(f'mAP@50-95   : {metrics.box.map:.4f}')
print(f'Precision   : {metrics.box.mp:.4f}')
print(f'Recall      : {metrics.box.mr:.4f}')
print()
print('Per-Klasse mAP@50:')
for cls, ap in zip(CLASSES, metrics.box.ap50):
    bar = chr(9608) * int(ap * 30)
    print(f'  {cls:22s}  {ap:.3f}  {bar}')

Lade: C:\Users\phili\Desktop\OTH\Master\fpp\hsp\hsp-1\Python\runs\rdd2022_yolo11m\weights\best.pt
Ultralytics 8.4.48  Python-3.11.15 torch-2.4.1+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
YOLO11n summary (fused): 101 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 37.656.2 MB/s, size: 238.8 KB)
val: Scanning C:\Users\phili\Desktop\OTH\Master\fpp\hsp\hsp-1\Python\data\yolo\labels\test... 8273 images, 8273 backgrounds, 0 corrupt: 92% ━━━━━━━━━━╸─ 8273/9035 919.1it/s 10.0s<0.8s


KeyboardInterrupt: 

In [ ]:
test_imgs = list((DATA_YOLO / 'images' / 'test').glob('*.jpg'))
samples = [str(p) for p in random.sample(test_imgs, min(6, len(test_imgs)))]

preds = model_eval.predict(source=samples, conf=0.25, iou=0.45,
                           device=0, save=False, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, r in zip(axes.flat, preds):
    ax.imshow(Image.open(r.path))
    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        color  = COLORS[cls_id % len(COLORS)]
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1, max(y1-4, 0), f'{CLASSES[cls_id]} {conf:.2f}',
                color=color, fontsize=7,
                bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.set_title(f'{Path(r.path).name} â€” {len(r.boxes)} Det.', fontsize=8)
    ax.axis('off')

plt.suptitle('Beispiel-Vorhersagen auf Test-Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

: 

: 

## 6. Zusammenfassung

In [ ]:
map50_col = 'metrics/mAP50(B)'
best_idx = df[map50_col].idxmax() if map50_col in df.columns else None

print('=' * 52)
print('  TRAINING ZUSAMMENFASSUNG')
print('=' * 52)
print(f'  Modell       : YOLOv11m')
print(f'  Datensatz    : RDD2022 ({sum(splits.values()):,} Bilder, 7 Klassen)')
print(f'  Epochen      : {len(df)}')
if best_idx is not None:
    print(f'  Beste Epoche : {int(df["epoch"].iloc[best_idx])}')
    print(f'  Bestes mAP50 : {df[map50_col].iloc[best_idx]:.4f}')
print(f'  Gewichte     : {best_weights}')
print()
print('  Test-Set:')
print(f'    mAP@50    : {metrics.box.map50:.4f}')
print(f'    mAP@50-95 : {metrics.box.map:.4f}')
print(f'    Precision : {metrics.box.mp:.4f}')
print(f'    Recall    : {metrics.box.mr:.4f}')

: 

: 